# ST-CDGM **V8** — run GPU complet

Mêmes jeux de données que depuis le début. Aucun prétraitement à lancer : les
**13 nœuds libres** sont dérivés à la volée depuis les 15 canaux bruts.

## Ce qui change par rapport à tous les runs précédents

| | avant | V8 |
|---|---|---|
| nœuds du DAG | plongements de métachemins, **mêmes entrées pour tous** | 13 quantités physiques, **un canal chacun** |
| drivers | un embedding partagé + un biais appris | **routage diagonal** : l'info inter-variable ne passe que par `A[u,v]` |
| structure | `A_dag` mono-lag | `A_inst` (τ=0) **et** `A_dag` (τ≥1) |
| prior | MSE vers la matrice complète | **C7** : 3 niveaux, λ annelé par niveau, hors-prior LIBRE |
| décodeur | requêtes apprises, aveugles à l'entrée | **A2a** : requêtes amorcées par l'état + encodage positionnel partagé |
| perte étage 1 | MSE sur log1p | **A3** : vraisemblance Bernoulli-Gamma, ancre `μ = p·α·β` |
| retour en mm | `expm1(μ)` | **A1** : `expm1(μ + s²/2)`, `s²` hétéroscédastique |

## Pourquoi ces changements

Trois audits ont trouvé le même défaut à trois endroits : **les variables ne se
distinguaient jamais par leurs entrées**, seulement par des poids appris. Les
types de nœuds du builder recevaient tous les mêmes 15 canaux ; `driver_encoder`
distribuait le même vecteur aux q variables ; et aucune ligne du dépôt ne
calculait l'IVT. Un DAG dans ces conditions est décoratif : rien ne le rend
load-bearing. V8 corrige les trois.

## Interrupteurs

Chaque brique s'éteint séparément (`V8` en Cell 2). P1 demande **un seul
changement par run** pour pouvoir attribuer l'effet. Tout à `False` ≈ pile V5.

## Cible

**Parité in-distribution + gain OOD.** Une régression ID de quelques pour cent
est prévue et acceptée : le DAG gelé est une feature OOD assumée, pas un
avantage ID. Le verdict se joue sur EC-Earth3, puis **une seule fois** sur le
holdout.

## Règle absolue

**NorESM2-MM est le holdout OOD.** Aucune cellule ne l'ouvre. Un garde lève à
la moindre tentative.

In [ ]:
# >>> Cell 1 : bootstrap Colab + montage Drive
import os, sys, json, math, time, warnings, subprocess
from pathlib import Path
warnings.filterwarnings("ignore")

GIT_URL    = "https://github.com/leonelkenfack/stcdgm.git"
GIT_BRANCH = "four-node-causal"
REPO_DIR   = "/content/climate_data"
DRIVE_ROOT = "/content/drive/MyDrive/climate_data"

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    # Les .nc d'entrainement (3 Go) sont gitignores : ils vivent sur Drive,
    # pas dans le depot. Sans ce montage, la Cell 2 s'arrete faute de donnees.
    if not Path("/content/drive").exists():
        from google.colab import drive
        drive.mount("/content/drive")

    if not Path(REPO_DIR).exists():
        # Clone sur le SSD local, jamais sur Drive : xarray y est ~20x plus lent.
        subprocess.check_call(["git", "clone", "--depth=200", "-b", GIT_BRANCH,
                               GIT_URL, REPO_DIR])
    else:
        subprocess.check_call(["git", "-C", REPO_DIR, "fetch", "origin"])
        subprocess.check_call(["git", "-C", REPO_DIR, "checkout", GIT_BRANCH])
        subprocess.check_call(["git", "-C", REPO_DIR, "pull", "origin", GIT_BRANCH])
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install",
                           "xbatcher", "omegaconf", "diffusers", "torch-geometric"])
    ROOT = Path(REPO_DIR)
else:
    ROOT = Path.cwd()

# chdir a la racine : les chemins relatifs (config/*.yaml, data/, checkpoints/)
# echouent sinon depuis /content.
os.chdir(ROOT)
for _p in (str(ROOT), str(ROOT / "src")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

import numpy as np, torch
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"racine  : {ROOT}")
print(f"torch {torch.__version__} | device = {DEVICE}")
if DEVICE.type == "cuda":
    _p = torch.cuda.get_device_properties(0)
    print(f"  {_p.name} | {_p.total_memory / 2**30:.1f} GiB")
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
else:
    print("  ATTENTION : prevu pour GPU. L'etage 2 sera tres lent sur CPU.")

In [ ]:
# >>> Cell 2 : configuration V8 + interrupteurs + garde holdout
from omegaconf import OmegaConf
from st_cdgm.data.derived import FREE_NODES

CONFIG = OmegaConf.load("config/training_config.yaml")

# --- Interrupteurs. Tout a False donne approximativement la pile V5.
#     P1 exige UN SEUL changement par run pour pouvoir attribuer l'effet.
V8 = OmegaConf.create(dict(
    free_nodes      = True,   # 13 noeuds libres derives au lieu des 15 bruts
    diagonal_driver = True,   # V7-M2 : chaque variable ne voit que son canal
    instantaneous   = True,   # C2/C5 : A(0) contemporaine en plus de A(tau>=1)
    edge_prior      = True,   # C7 : prior 3 niveaux, annele par niveau
    spatial_queries = True,   # A2a : requetes du decodeur dependantes de l'entree
    bernoulli_gamma = True,   # A3 : vraisemblance BG, ancre mu = p*alpha*beta
    jensen          = True,   # A1 : correction du retour log1p -> mm
))
print(OmegaConf.to_yaml(V8))

# --- Budget. Pour un smoke, reduire EPOCHS ; ne JAMAIS toucher aux seuils.
EPOCHS_S1   = int(os.environ.get("V8_EPOCHS_S1", 30))
EPOCHS_S2   = int(os.environ.get("V8_EPOCHS_S2", 20))
N_EVAL      = int(os.environ.get("V8_N_EVAL", 300))
K_ENSEMBLE  = int(os.environ.get("V8_ENSEMBLE", 16))
SEQ_LEN     = int(CONFIG.data.seq_len)
HIDDEN      = int(CONFIG.rcn.hidden_dim)
LR_SHAPE    = (23, 26)
HR_SHAPE    = (172, 179)
CKPT_DIR    = Path("checkpoints_v8"); CKPT_DIR.mkdir(exist_ok=True)
RESULTS_DIR = Path("results"); RESULTS_DIR.mkdir(exist_ok=True)

# --- Donnees : EXACTEMENT celles utilisees jusqu'ici, rien de plus.
# Les .nc d'entrainement sont gitignores (3 Go) : ils viennent de Drive sur
# Colab, du depot en local. Le statique, lui, EST suivi par git.
def resolve(*candidates):
    for c in candidates:
        if Path(c).exists():
            return Path(c)
    return None

LR_PATH = resolve(
    "data/raw/train/predictor_ACCESS-CM2_hist.nc",
    f"{DRIVE_ROOT}/data/train/predictor_ACCESS-CM2_hist.nc",
    f"{DRIVE_ROOT}/data/raw/train/predictor_ACCESS-CM2_hist.nc",
    f"{DRIVE_ROOT}/predictor_ACCESS-CM2_hist.nc")
HR_PATH = resolve(
    "data/raw/train/pr_ACCESS-CM2_hist.nc",
    f"{DRIVE_ROOT}/data/train/pr_ACCESS-CM2_hist.nc",
    f"{DRIVE_ROOT}/data/raw/train/pr_ACCESS-CM2_hist.nc",
    f"{DRIVE_ROOT}/pr_ACCESS-CM2_hist.nc")
STATIC_PATH = resolve(
    "data/raw/static_predictors/ERA5_eval_ccam_12km.198110_NZ_Invariant.nc",
    f"{DRIVE_ROOT}/data/raw/static_predictors/ERA5_eval_ccam_12km.198110_NZ_Invariant.nc")

_manquants = [n for n, v in (("predictor_ACCESS-CM2_hist.nc", LR_PATH),
                             ("pr_ACCESS-CM2_hist.nc", HR_PATH),
                             ("ERA5_..._NZ_Invariant.nc", STATIC_PATH)) if v is None]
if _manquants:
    raise FileNotFoundError(
        "Fichiers introuvables : " + ", ".join(_manquants)
        + os.linesep
        + "Les .nc d'entrainement sont gitignores (3 Go). Les deposer dans "
        + f"{DRIVE_ROOT}/data/train/ ou, hors Colab, dans data/raw/train/ "
        + "a la racine du depot.")

# --- Garde holdout. NorESM2-MM est pre-enregistre comme intouchable : une
#     seule evaluation finale, jamais pendant le developpement.
HOLDOUT = "NorESM2"

def guard(p):
    assert HOLDOUT not in str(p), f"STOP - {p} touche le holdout {HOLDOUT}-MM."
    return Path(p)

for _p in (LR_PATH, HR_PATH, STATIC_PATH):
    guard(_p)

print(f"LR      : {LR_PATH}")
print(f"HR      : {HR_PATH}")
print(f"statique: {STATIC_PATH}")
print("13 noeuds :", list(FREE_NODES))

In [ ]:
# >>> Cell 3 : pipeline - les 13 noeuds derives a la volee
from st_cdgm.data.pipeline import NetCDFDataPipeline

# lr_free_nodes=True : la derivation est faite UNE fois sur le jeu complet, pas
# par fenetre. Tout l'aval (_dataset_to_numpy, lr_grid_to_nodes, le driver du
# RCN) voit alors 13 canaux dans l'ordre de FREE_NODES, ce qui est exactement
# la carte identite du routage diagonal.
t0 = time.time()
pipeline = NetCDFDataPipeline(
    lr_path=guard(LR_PATH), hr_path=guard(HR_PATH), static_path=guard(STATIC_PATH),
    seq_len=SEQ_LEN,
    baseline_strategy=str(CONFIG.data.baseline_strategy),
    baseline_factor=int(CONFIG.data.baseline_factor),
    normalize=bool(CONFIG.data.normalize),
    nan_fill_strategy=str(CONFIG.data.nan_fill_strategy),
    precipitation_delta=float(CONFIG.data.precipitation_delta),
    lr_free_nodes=bool(V8.free_nodes),
    lr_variables=None if V8.free_nodes else list(CONFIG.data.lr_variables),
    hr_variables=list(CONFIG.data.hr_variables),
    static_variables=(list(CONFIG.data.static_variables)
                      if CONFIG.data.get("static_variables") else []),
    train_start_date=CONFIG.data.get("train_start_date"),
    train_end_date=CONFIG.data.get("train_end_date"),
    val_start_date=CONFIG.data.get("val_start_date"),
    val_end_date=CONFIG.data.get("val_end_date"),
    test_start_date=CONFIG.data.get("test_start_date"),
    test_end_date=CONFIG.data.get("test_end_date"),
)
LR_VARS = list(pipeline.get_lr_dataset().data_vars)
print(f"[{time.time() - t0:.0f}s] canaux LR ({len(LR_VARS)}) : {LR_VARS}")

if V8.free_nodes:
    # L'ordre est load-bearing : une permutation silencieuse ferait tourner le
    # modele en associant chaque variable au mauvais champ, sans rien signaler.
    assert LR_VARS == list(FREE_NODES), (
        f"ordre des canaux != FREE_NODES\n  recu   : {LR_VARS}\n  attendu: {list(FREE_NODES)}")

train_dataset = pipeline.build_sequence_dataset(split="train", training=True)
val_dataset   = pipeline.build_sequence_dataset(split="val")
test_dataset  = pipeline.build_sequence_dataset(split="test")

_s = next(iter(train_dataset))
print("echantillon : lr", tuple(_s["lr"].shape),
      "| residual", tuple(_s["residual"].shape),
      "| baseline", tuple(_s["baseline"].shape))
assert _s["lr"].shape[1] == len(LR_VARS)

In [ ]:
# >>> Cell 4 : la pile V8
from st_cdgm.models.graph_builder import HeteroGraphBuilder
from st_cdgm.models.intelligible_encoder import (
    IntelligibleVariableEncoder, IntelligibleVariableConfig)
from st_cdgm.models.causal_rcn import RCNCell, RCNSequenceRunner
from st_cdgm.models.regression_head import GraphToGridDecoder
from st_cdgm.models.bernoulli_gamma import BernoulliGammaHead
from st_cdgm.priors import load_edge_prior, DEFAULT_LEVEL_FLOORS

torch.manual_seed(SEED)

# --- graphe : 13 types de noeuds REELLEMENT distincts ----------------------
# free_nodes_v8 remplace entierement le jeu de noeuds (pas de GP850/500/250).
builder = HeteroGraphBuilder(
    lr_shape=LR_SHAPE, hr_shape=HR_SHAPE,
    static_dataset=pipeline.get_static_dataset(),
    include_mid_layer=not bool(V8.free_nodes),
    free_nodes_v8=bool(V8.free_nodes),
)
hetero_template, _report = builder.build()
NODE_TYPES = list(builder.dynamic_node_types)
Q = len(NODE_TYPES)
print(f"q = {Q} variables : {NODE_TYPES}")

# --- encodeur : un metachemin spatial par variable -------------------------
enc_cfgs = [IntelligibleVariableConfig(name=f"{n}_spat",
                                       meta_path=(n, "spat_adj", n), pool="mean")
            for n in NODE_TYPES]
encoder = IntelligibleVariableEncoder(
    configs=enc_cfgs, hidden_dim=HIDDEN,
    conditioning_dim=int(CONFIG.encoder.conditioning_dim)).to(DEVICE)

# MATERIALISATION OBLIGATOIRE. L'encodeur est bati sur des LazyModule : ses
# poids n'existent qu'apres un premier forward, et il faut un graphe PORTANT
# DES FEATURES (le template n'en a pas). Les notebooks precedents ne s'en
# apercevaient pas parce qu'ils chargeaient un checkpoint - load_state_dict
# materialise. V8 part de zero : sans ce forward a blanc, encoder.parameters()
# serait VIDE au moment de construire l'optimiseur, et l'encodeur resterait a
# son initialisation aleatoire pendant tout l'entrainement, EN SILENCE.
from st_cdgm.evaluation.evaluation_xai import convert_sample_to_batch

with torch.no_grad():
    _warm = convert_sample_to_batch(_s, builder, DEVICE)
    _H0 = encoder.init_state(_warm["hetero"])
_n_enc = sum(p.numel() for p in encoder.parameters())
assert _n_enc > 0, "encodeur non materialise : l'optimiseur serait vide"
assert _H0.shape[0] == Q, f"H_init a {_H0.shape[0]} variables, attendu {Q}"
print(f"encodeur  : {_n_enc:,} parametres materialises | H_init {tuple(_H0.shape)}")

# --- prior C7 --------------------------------------------------------------
edge_prior, A_prior, A_inst_prior = None, None, None
if V8.edge_prior and V8.free_nodes:
    edge_prior = load_edge_prior()
    assert len(edge_prior.nodes) == Q, (
        f"prior sur {len(edge_prior.nodes)} noeuds, graphe a {Q} variables")
    # node_order : l'ordre du RCN n'a aucune raison d'etre celui du YAML.
    # FILTRAGE PAR LAG obligatoire : A_dag est A(1), A_inst est A(0). Initialiser
    # A(1) avec TOUTES les aretes y injecterait les 24 aretes contemporaines que
    # la perte route pourtant vers A(0) — et A(0) demarrerait au bruit, sans
    # prior, alors que c'est la structure que C2/C5 existe pour exprimer.
    A_prior = torch.as_tensor(edge_prior.matrix(lag=1, node_order=NODE_TYPES))
    A_inst_prior = (torch.as_tensor(edge_prior.matrix(lag=0, node_order=NODE_TYPES))
                    if V8.instantaneous else None)
    print(f"prior C7  : {len(edge_prior)} aretes {edge_prior.by_level()}")
    print(f"            orientation {edge_prior.by_orient()}")
    print(f"            init : A(1) <- {len(edge_prior.edge_list(lag=1))} aretes | "
          f"A(0) <- {len(edge_prior.edge_list(lag=0))} aretes")
    if not V8.instantaneous and edge_prior.edge_list(lag=0):
        raise ValueError(
            f"{len(edge_prior.edge_list(lag=0))} aretes du prior sont a lag 0 "
            f"mais V8.instantaneous=False : elles seraient ecrasees sur A(1).")

# --- RCN : routage diagonal (V7-M2) + A(0) contemporaine -------------------
rcn_cell = RCNCell(
    num_vars=Q, hidden_dim=HIDDEN, driver_dim=len(LR_VARS),
    reconstruction_dim=len(LR_VARS),
    dropout=float(CONFIG.rcn.dropout),
    dag_prior=A_prior, inst_prior=A_inst_prior,
    instantaneous=bool(V8.instantaneous),
    driver_routing=("diagonal" if (V8.diagonal_driver and V8.free_nodes) else "shared"),
).to(DEVICE)
rcn_runner = RCNSequenceRunner(rcn_cell, detach_interval=CONFIG.rcn.get("detach_interval"))
print(f"RCN       : routage={rcn_cell.driver_routing} | "
      f"A(0)={'oui' if rcn_cell.A_inst is not None else 'non'}")

# --- decodeur : requetes spatiales (A2a) -----------------------------------
_rh = CONFIG.two_stage.regression_head
regression_head = GraphToGridDecoder(
    d_model=HIDDEN, hr_h=HR_SHAPE[0], hr_w=HR_SHAPE[1],
    intermediate_h=int(_rh.intermediate_h), intermediate_w=int(_rh.intermediate_w),
    n_heads=int(_rh.n_heads), refine_channels=int(_rh.refine_channels),
    query_mode=("spatial" if V8.spatial_queries else "learned"),
    lr_h=LR_SHAPE[0], lr_w=LR_SHAPE[1],
).to(DEVICE)
print(f"decodeur  : query_mode={regression_head.query_mode} | "
      f"features={regression_head.feature_channels}")

# --- tete Bernoulli-Gamma (A3) --------------------------------------------
bg_head = (BernoulliGammaHead(regression_head.feature_channels).to(DEVICE)
           if V8.bernoulli_gamma else None)
# A3 exclut le melange convexe du skip-block : A4 propose de le retirer, et il
# casserait l'interpretation de mu = p*alpha*beta comme moyenne conditionnelle.
skip_block = None

STAGE1_MODULES = [encoder, rcn_cell, regression_head] + ([bg_head] if bg_head else [])
n_par = sum(p.numel() for m in STAGE1_MODULES for p in m.parameters())
print(f"parametres etage 1 : {n_par:,}")

In [ ]:
# >>> Cell 5 : entrainement etage 1
from torch.optim import AdamW
from st_cdgm.training.training_loop import train_epoch_stage1

def iterate_batches(ds, bld=None, dev=None):
    """Signature (data_loader, builder, device) attendue par les helpers."""
    bld = bld if bld is not None else builder
    dev = dev if dev is not None else DEVICE
    for s in ds:
        yield [convert_sample_to_batch(s, bld, dev)]

# bg_head fait partie des parametres optimises : l'oublier laisserait la tete
# a son initialisation et la NLL ne descendrait jamais. L'encodeur, lui, a ete
# materialise en Cell 4 - sinon ses poids lazy seraient absents d'ici.
assert all(sum(1 for _ in m.parameters()) > 0 for m in STAGE1_MODULES),     "un module n'expose aucun parametre - voir la materialisation en Cell 4"
opt_s1 = AdamW([p for m in STAGE1_MODULES for p in m.parameters()],
               lr=float(CONFIG.training.learning_rate), betas=(0.9, 0.99))

history, best = [], float("inf")
for ep in range(EPOCHS_S1):
    t_ep = time.time()
    m = train_epoch_stage1(
        encoder=encoder, rcn_runner=rcn_runner, regression_head=regression_head,
        optimizer=opt_s1, data_loader=iterate_batches(train_dataset),
        device=DEVICE, epoch_idx=ep,
        lambda_reg=float(CONFIG.two_stage.get("lambda_reg", 1.0)),
        beta_rec=float(CONFIG.two_stage.get("beta_rec", 0.05)),
        gamma_dag_max=float(CONFIG.two_stage.get("gamma_dag_max", 0.10)),
        lambda_l1=float(CONFIG.two_stage.get("lambda_l1", 0.01)),
        gradient_clipping=1.0, use_amp=(DEVICE.type == "cuda"),
        # --- A3 : la NLL Bernoulli-Gamma remplace la MSE ------------------
        bg_head=bg_head,
        # --- C7 : prior 3 niveaux, annele PAR NIVEAU ----------------------
        edge_prior=edge_prior, prior_node_order=NODE_TYPES,
        prior_level_floors=DEFAULT_LEVEL_FLOORS, prior_anneal_epochs=EPOCHS_S1,
        # --- exclusions imposees par bg_head (le loop les verifie) --------
        skip_block=skip_block, p1_tail_alpha=0.0, p3_k_samples=0,
        verbose=(ep == 0),
    )
    m["epoch"] = ep
    m["seconds"] = round(time.time() - t_ep, 1)
    history.append(m)
    print(f"[S1 {ep + 1:2d}/{EPOCHS_S1}] loss={m['loss']:.5f} "
          f"reg={m['loss_reg']:.5f} dag={m.get('loss_dag', 0.0):.4f} "
          f"({m['seconds']:.0f}s)")
    if m["loss"] < best:
        best = m["loss"]
        torch.save({"epoch": ep,
                    "encoder_state_dict": encoder.state_dict(),
                    "rcn_cell_state_dict": rcn_cell.state_dict(),
                    "regression_head_state_dict": regression_head.state_dict(),
                    **({"bg_head_state_dict": bg_head.state_dict()} if bg_head else {}),
                    "node_types": NODE_TYPES,
                    "v8": OmegaConf.to_container(V8)},
                   CKPT_DIR / "stage1_best.pth")

json.dump(history, open(RESULTS_DIR / "v8_stage1_history.json", "w"),
          indent=2, default=float)
print(f"meilleure perte etage 1 : {best:.5f}")

In [ ]:
# >>> Cell 6 : ce que le DAG a appris - decouvert contre impose
A_lag = rcn_cell.dag_matrix(masked=True).detach().cpu().numpy()
print(f"A(tau>=1) : norme={np.linalg.norm(A_lag):.4f} "
      f"asymetrie={np.abs(A_lag - A_lag.T).mean():.4f}")
if rcn_cell.A_inst is not None:
    A0 = rcn_cell.dag_matrix(masked=True, lag=0).detach().cpu().numpy()
    print(f"A(0)      : norme={np.linalg.norm(A0):.4f} "
          f"asymetrie={np.abs(A0 - A0.T).mean():.4f}")

def _score(A, lag_v, label):
    """Confronte UN operateur a la partie du prior qui lui revient.
    Melanger les lags rendrait le bilan faux sur 24 des 33 aretes."""
    P = edge_prior.matrix(lag=lag_v, node_order=NODE_TYPES)
    support = P != 0
    if not support.any():
        return None
    # Seuil sur le quantile 80 : on garde le meme budget d'aretes que le prior
    # plutot qu'un seuil absolu arbitraire qui dependrait de l'echelle de A.
    thr = float(np.quantile(np.abs(A), 0.80))
    found = np.abs(A) > thr
    kept, dropped = int((found & support).sum()), int((~found & support).sum())
    novel = int((found & ~support).sum())
    print(f"\nprior : {int(support.sum())} aretes | seuil |A| > {thr:.4f}")
    print(f"  retenues par les donnees : {kept}")
    print(f"  REJETEES                 : {dropped}   <- signal de decouverte")
    print(f"                                            NEGATIF, a rapporter (C7)")
    print(f"  hors prior               : {novel}   <- decouverte au-dela du prior")

    # Par niveau : le niveau 3 DOIT pouvoir tomber. C'est tout l'objet de
    # l'annealing par niveau - un prior speculatif que les donnees ne peuvent
    # pas rejeter n'est plus un prior, c'est une contrainte.
    print("\n  survie par niveau de credibilite :")
    per_level = {}
    for lvl, mask in edge_prior.level_masks(node_order=NODE_TYPES).items():
        mask = mask & support        # restreindre au lag de CET operateur
        if mask.any():
            r = float((found & mask).sum() / mask.sum())
            per_level[int(lvl)] = r
            print(f"    niveau {lvl} ({int(mask.sum()):2d} aretes) : {100 * r:4.0f} %")

    # Les aretes rejetees, nommees : c'est le livrable scientifique de C7.
    rejected = [(NODE_TYPES[i], NODE_TYPES[j])
                for i, j in zip(*np.where(~found & support))]
    if rejected:
        print("\n  aretes du prior REJETEES par les donnees :")
        for a, b in rejected:
            print(f"    {a} -> {b}")

    return {"operator": label, "n_prior": int(support.sum()), "kept": kept,
            "dropped": dropped, "novel": novel, "threshold": thr,
            "survival_by_level": per_level, "rejected_edges": rejected}


if edge_prior is not None:
    report = [r for r in (_score(A_lag, 1, "A(tau>=1)"),
                          (_score(A0, 0, "A(0)") if rcn_cell.A_inst is not None else None))
              if r is not None]
    json.dump(report, open(RESULTS_DIR / "v8_dag_vs_prior.json", "w"), indent=2)

In [ ]:
# >>> Cell 7 : A1 - calibration de Jensen + audit de conformite de l'etage 1
from st_cdgm.evaluation.jensen import JensenCorrector
from st_cdgm.training.stage1_paths import predict_mu_hr

for _m in STAGE1_MODULES:
    _m.eval()

@torch.no_grad()
def collect(ds, n_max=400):
    """mu / cible / baseline en espace log1p, [N, H, W]."""
    mus, tgs, bls = [], [], []
    for i, s in enumerate(ds):
        if i >= n_max:
            break
        b = convert_sample_to_batch(s, builder, DEVICE)
        t = b["residual"][-1].to(DEVICE)
        if t.dim() == 3:
            t = t.unsqueeze(0)
        bl = b["baseline"][-1].to(DEVICE)
        if bl.dim() == 3:
            bl = bl.unsqueeze(0)
        mu = predict_mu_hr(b, variant="causal", encoder=encoder, rcn_runner=rcn_runner,
                           regression_head=regression_head, builder=builder,
                           device=DEVICE, target_shape=t.shape[-2:], bg_head=bg_head)
        mus.append(mu.reshape(t.shape[-2:]).cpu())
        tgs.append(t.reshape(t.shape[-2:]).cpu())
        bls.append(bl.reshape(t.shape[-2:]).cpu())
    return (torch.stack(mus).numpy(), torch.stack(tgs).numpy(), torch.stack(bls).numpy())

mu_tr, tg_tr, bl_tr = collect(train_dataset)
print("echantillon de calibration :", mu_tr.shape)

# s^2 calibre sur le TRAIN uniquement. C'est un parametre de calibration :
# l'estimer sur le test ferait fuiter la cible dans la metrique.
jc = JensenCorrector.fit(mu_tr, tg_tr, bl_tr) if V8.jensen else None
if jc is not None:
    print(jc)
    torch.save(jc.state_dict(), CKPT_DIR / "jensen.pt")

# --- Audit de conformite. C1 juge le biais APRES correction de Jensen :
# sinon il mesurerait l'inegalite de Jensen et non le modele. Un etage 1
# EXACT en log1p affiche ~22 % de biais conditionnel sans la correction.
mu_te, tg_te, bl_te = collect(test_dataset, n_max=N_EVAL)
x_mm = np.expm1(np.clip(bl_te + tg_te, -20, 20))

def cond_bias(pred_mm):
    q = np.unique(np.quantile(pred_mm, np.linspace(0, 1, 11)))
    bid = np.clip(np.digitize(pred_mm.ravel(), q[1:-1]), 0, len(q) - 2)
    xr_, mr_ = x_mm.ravel(), pred_mm.ravel()
    return [100 * (xr_[bid == k].mean() - mr_[bid == k].mean())
            / max(xr_[bid == k].mean(), 1e-9)
            for k in range(len(q) - 1) if (bid == k).sum() > 500]

naive_mm = np.expm1(np.clip(bl_te + mu_te, -20, 20))
corr_mm = jc.to_mm(mu_te, bl_te, delta=0.0).numpy() if jc is not None else naive_mm
b_naive = max(map(abs, cond_bias(naive_mm)))
b_corr = max(map(abs, cond_bias(corr_mm)))
print(f"\nbiais conditionnel max : naif {b_naive:5.1f} %  ->  corrige {b_corr:5.1f} %")
print(f"C1 {'PASS' if b_corr < 5.0 else 'FAIL'} (seuil 5 %) | "
      f"part imputable a Jensen : {b_naive - b_corr:.1f} points")

json.dump({"C1_bias_corrected_pct": float(b_corr),
           "C1_bias_naive_pct": float(b_naive),
           "C1_jensen_share_pct": float(b_naive - b_corr),
           "C1_pass": bool(b_corr < 5.0)},
          open(RESULTS_DIR / "v8_stage1_audit.json", "w"), indent=2)

In [ ]:
# >>> Cell 8 : gel de l'etage 1 + cache pour l'etage 2
from st_cdgm.training.two_stage import freeze_stage1, precompute_stage1_outputs
from st_cdgm.training.stage1_paths import calibrate_sigma_data_variant

sig = calibrate_sigma_data_variant(
    variant="causal", regression_head=regression_head, data_loader=train_dataset,
    iterate_batches_fn=iterate_batches, builder=builder, device=DEVICE,
    encoder=encoder, rcn_runner=rcn_runner, max_samples=200)
SIGMA_DATA = float(sig["sigma_data"])
print(f"sigma_data = {SIGMA_DATA:.5f}")

freeze_stage1(encoder, rcn_cell, regression_head, *([bg_head] if bg_head else []))
rcn_cell.A_dag.requires_grad_(False)
if rcn_cell.A_inst is not None:
    rcn_cell.A_inst.requires_grad_(False)
print("etage 1 gele, A_dag et A(0) compris - le DAG devient une feature OOD assumee")

# L'ancre mise en cache est mu = p*alpha*beta reexprimee en residu log1p,
# pas la projection 1 canal du decodeur (qui n'est plus entrainee sous A3).
cache = precompute_stage1_outputs(
    encoder=encoder, rcn_runner=rcn_runner, regression_head=regression_head,
    train_dataset=train_dataset,
    iterate_batches_fn=lambda s: convert_sample_to_batch(s, builder, DEVICE),
    device=DEVICE, bg_head=bg_head)
print({k: tuple(v.shape) for k, v in cache.items()})

torch.save({"sigma_data": SIGMA_DATA, "node_types": NODE_TYPES,
            "v8": OmegaConf.to_container(V8)},
           CKPT_DIR / "stage1_frozen_meta.pth")

In [ ]:
# >>> Cell 9 : etage 2 - diffusion EDM sur le residu
from torch.utils.data import TensorDataset, DataLoader
from st_cdgm.models.diffusion_decoder import CausalDiffusionDecoder
from st_cdgm.training.two_stage import train_epoch_stage2_cached

torch.manual_seed(SEED)
diffusion = CausalDiffusionDecoder(
    height=HR_SHAPE[0], width=HR_SHAPE[1],
    conditioning_channels=int(CONFIG.diffusion.get("conditioning_channels", 2)),
    sigma_data=SIGMA_DATA,
).to(DEVICE)
print(f"parametres etage 2 : {sum(p.numel() for p in diffusion.parameters()):,}")

cached = TensorDataset(cache["mu_HR"], cache["baseline_log"],
                       cache["delta_target"], cache["valid_mask"])
BS = int(CONFIG.training.get("batch_size", 8))

def cached_loader():
    for mu, bl, dt, vm in DataLoader(cached, batch_size=BS, shuffle=True, drop_last=True):
        yield {"mu_HR": mu.to(DEVICE), "baseline_log": bl.to(DEVICE),
               "delta_target": dt.to(DEVICE), "valid_mask": vm.to(DEVICE)}

opt_s2 = torch.optim.AdamW(
    diffusion.parameters(),
    lr=float(CONFIG.training.get("learning_rate_stage2", CONFIG.training.learning_rate)),
    betas=(0.9, 0.99))

hist2 = []
for ep in range(EPOCHS_S2):
    t_ep = time.time()
    m2 = train_epoch_stage2_cached(
        diffusion_decoder=diffusion, optimizer=opt_s2,
        cached_dataloader=cached_loader(), device=DEVICE,
        use_amp=(DEVICE.type == "cuda"), gradient_clipping=1.0,
        # conditioning dropout : entraine la branche inconditionnelle, requise
        # pour cfg_scale > 1 a l'inference (CorrDiff, Mardani 2024 sec 4.2).
        conditioning_dropout_prob=0.13,
        verbose=(ep == 0))
    m2 = dict(m2); m2["epoch"] = ep; m2["seconds"] = round(time.time() - t_ep, 1)
    hist2.append(m2)
    print(f"[S2 {ep + 1:2d}/{EPOCHS_S2}] {m2}")
    torch.save({"epoch": ep, "diffusion_state_dict": diffusion.state_dict(),
                "sigma_data": SIGMA_DATA}, CKPT_DIR / "stage2_last.pth")

json.dump(hist2, open(RESULTS_DIR / "v8_stage2_history.json", "w"),
          indent=2, default=float)

In [ ]:
# >>> Cell 10 : evaluation + verdict pre-enregistre
from st_cdgm.evaluation.eval_metrics_dual_convention import (
    to_mm_day, compute_f1_both_conventions)
from st_cdgm.evaluation.two_stage_inference import sample_once_edm

diffusion.eval()
preds, truths = [], []
with torch.no_grad():
    for i, s in enumerate(test_dataset):
        if i >= N_EVAL:
            break
        b = convert_sample_to_batch(s, builder, DEVICE)
        t = b["residual"][-1].to(DEVICE)
        if t.dim() == 3:
            t = t.unsqueeze(0)
        bl = b["baseline"][-1].to(DEVICE)
        if bl.dim() == 3:
            bl = bl.unsqueeze(0)
        mu = predict_mu_hr(b, variant="causal", encoder=encoder, rcn_runner=rcn_runner,
                           regression_head=regression_head, builder=builder,
                           device=DEVICE, target_shape=t.shape[-2:], bg_head=bg_head)
        members = [sample_once_edm(diffusion, mu, bl, device=DEVICE)
                   for _ in range(K_ENSEMBLE)]
        # expm1 PAR MEMBRE puis moyenne. L'ordre inverse rouvrirait un ecart de
        # Jensen que A1 ne corrige pas : A1 porte sur la moyenne conditionnelle,
        # pas sur des tirages. Mesure sur ces donnees : -2 a -15 % au p99.
        preds.append(torch.stack([to_mm_day(bl + mu + d) for d in members]).mean(0).cpu())
        truths.append(to_mm_day(bl + t).cpu())
        if (i + 1) % 50 == 0:
            print(f"  {i + 1}/{N_EVAL}")

pred = torch.cat(preds).squeeze()
truth = torch.cat(truths).squeeze()
clim99 = torch.quantile(truth.float(), 0.99, dim=0)
clim95 = torch.quantile(truth.float(), 0.95, dim=0)
f1 = compute_f1_both_conventions(pred, truth, clim99, clim95)
rmse = float(((pred - truth) ** 2).mean().sqrt())
print(f"\nF1@p99 : {f1}")
print(f"RMSE   : {rmse:.4f} mm/j")

# References mesurees dans les runs precedents, in-protocol.
REF = {"v5_per_gridpoint": 0.841, "noncausal_per_gridpoint": 0.816,
       "v5_pooled": 0.512, "corrdiff_pooled": 0.550}
verdict = {
    "f1": f1, "rmse_mm": rmse, "references": REF,
    "ensemble_K": K_ENSEMBLE, "n_eval": N_EVAL,
    "v8": OmegaConf.to_container(V8),
    "lecture": (
        "Cible V8 = PARITE in-distribution + gain OOD, pas un gain ID. Une "
        "regression ID de quelques pour cent est PREVUE et acceptee : le DAG "
        "gele est une feature OOD assumee (critique froide 2026-07-05). Le "
        "verdict se joue sur EC-Earth3, puis UNE SEULE FOIS sur le holdout "
        "NorESM2-MM. Un gain ID ici serait une bonne surprise, pas le critere."),
}
json.dump(verdict, open(RESULTS_DIR / "v8_verdict.json", "w"), indent=2, default=float)
print("\n=== ECRIT results/v8_verdict.json ===")
print(verdict["lecture"])